# KYC / bank-statement trial — main.ipynb
Run top-to-bottom. Helpers live in `utils.py` (functions only). Outputs go to `outputs/` (gitignored).

**Kernel:** `.venv` from the README (Python 3.11 or 3.12). One environment runs every cell.
**Run on Windows.** See README. Keyword classification is the classifier. No Hugging Face, no language model.
No Hugging Face, no cloud APIs. Everything runs on CPU locally.

## 0. Setup / paths

In [ ]:
import os
from pathlib import Path
import json, time
import numpy as np, pandas as pd

# Table weights come from Baidu BOS, not Hugging Face. Set this before paddleocr is imported.
os.environ["PADDLE_PDX_MODEL_SOURCE"] = "bos"
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"
os.environ["HF_HUB_OFFLINE"] = "1"

import utils as U

pdfs = sorted(Path("data").glob("*.pdf"))
if not pdfs:
    raise SystemExit("Put a PDF in data\\ and run this cell again.")
PDF = pdfs[0]                               # if data\\ has several PDFs, set PDF = Path("data/your_file.pdf")
if len(pdfs) > 1:
    print("Several PDFs in data/. Using", PDF, "— set PDF = Path(...) to choose another.")
LEAD_ID = PDF.stem                          # primary key; replace with extracted application ID when applicant pages exist
OUT = Path("outputs") / LEAD_ID
OUT.mkdir(parents=True, exist_ok=True)
DPI = 200
print(PDF, PDF.exists(), OUT)

## 1. Ingest PDF → page count + preview

In [ ]:
pages_rgb = U.render_pages(PDF, dpi=DPI)
print("pages:", len(pages_rgb), {i: im.shape for i, im in pages_rgb.items()})

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 8))
for a, i in zip(ax, [0, len(pages_rgb) - 1]):
    a.imshow(pages_rgb[i]); a.set_title(f"page {i+1}"); a.axis("off")
plt.show()

## 2. Preprocess (grayscale + optional deskew)
Binarisation is intentionally off (it hurt OCR in the benchmark).

In [ ]:
DO_DESKEW = True
pages_gray, angles = {}, {}
for i, im in pages_rgb.items():
    g = U.to_gray(im)
    if DO_DESKEW:
        g, angles[i] = U.deskew(g)
    pages_gray[i] = g
print("deskew angles:", angles)

## 3. OCR (RapidOCR) → JSON dump per page

In [ ]:
ocr = {}
t0 = time.time()
for i, g in pages_gray.items():
    ocr[i] = U.ocr_page(g)
    U.save_json(ocr[i], OUT / f"ocr_p{i+1}.json")
print(f"{time.time()-t0:.1f}s for {len(ocr)} pages")

In [ ]:
page_lines = {i: U.lines_from_words(w) for i, w in ocr.items()}
page_text = {i: "\n".join(l) for i, l in page_lines.items()}
print(page_text[0][:1500])

## 4. Page classification — keywords
Three classes per page: `applicant` | `statement` | `not_needed`.
Word lists are editable here (generic, not bank-specific). A page goes to the best-scoring class if score ≥ 0.35, else `not_needed`.

In [ ]:
KEYWORDS = {
    "statement": ["statement", "opening balance", "closing balance", "balance", "debit", "credit",
                  "cr/dr", "dr/cr", "withdrawal", "deposit", "narration", "description", "value date",
                  "txn", "transaction", "cheque", "neft", "imps", "upi", "rtgs", "available balance"],
    "applicant": ["application", "applicant", "co-applicant", "application id", "date of birth", "dob",
                  "pan", "aadhaar", "kyc", "address proof", "occupation", "father", "signature of applicant",
                  "loan amount", "nominee", "marital status", "gender"],
}

In [ ]:
kw = {i: U.keyword_classify(t, KEYWORDS) for i, t in page_text.items()}
kw_df = pd.DataFrame([{"page": i+1, "kw_label": l, **{f"kw_{k}": v for k, v in s.items()}} for i, (l, s) in kw.items()])
kw_df

## 5. Page classification — BERT is not used
This package does not download or run a language model. Keyword labels from step 4 are the classifier.
The cell below only runs if you already have a local model folder. Leave that folder absent. Do not install `torch` or `transformers`.

In [ ]:
BERT_DIR = Path("models/distilbert-base-uncased")
PROTOTYPES = {
    "statement": ["account statement opening balance closing balance date description debit credit balance",
                  "value date narration cheque no withdrawal deposit available balance NEFT UPI IMPS"],
    "applicant": ["loan application form applicant name co-applicant date of birth PAN aadhaar address",
                  "application id customer details KYC occupation signature of applicant"],
    "not_needed": ["terms and conditions declaration disclaimer", "blank page", "photograph identity card"],
}
bert = {}
if BERT_DIR.exists():
    try:
        res = U.bert_classify([page_text[i] for i in sorted(page_text)], PROTOTYPES, BERT_DIR)
        bert = dict(zip(sorted(page_text), res))
    except ImportError as e:
        print("BERT skipped (pip install torch transformers):", e)
else:
    print("BERT skipped: no local model at", BERT_DIR)
bert_df = pd.DataFrame([{"page": i+1, "bert_label": l, **{f"bert_{k}": v for k, v in s.items()}} for i, (l, s) in bert.items()])
bert_df

## 6. Combine classifiers
Rule: keyword label wins; BERT used only when keyword says `not_needed` (fallback).

In [ ]:
cls = kw_df[["page", "kw_label"]].copy()
cls["bert_label"] = cls["page"].map({r.page: r.bert_label for r in bert_df.itertuples()}) if len(bert_df) else None
cls["label"] = [k if k != "not_needed" or not isinstance(b, str) else b for k, b in zip(cls.kw_label, cls.bert_label)]
statement_pages = [p - 1 for p in cls.page[cls.label == "statement"]]
applicant_pages = [p - 1 for p in cls.page[cls.label == "applicant"]]
print("statement:", [p+1 for p in statement_pages], "applicant:", [p+1 for p in applicant_pages])
cls

## 7. Locate applicant / account-details text
Use applicant pages if any; otherwise the header block of the first statement page (lines above the table header row).

In [ ]:
def header_block(lines):
    for j, ln in enumerate(lines):
        l = ln.lower()
        if sum(k in l for k in ["date", "description", "balance", "amount", "narration", "debit", "credit"]) >= 3:
            return lines[:j]
    return lines[:40]

src_pages = applicant_pages or statement_pages[:1]
details_lines = [ln for p in src_pages for ln in (page_lines[p] if p in applicant_pages else header_block(page_lines[p]))]
print("source pages:", [p+1 for p in src_pages]); print("\n".join(details_lines))

## 8. Parse applicant fields (Customer ID, Opening / Closing Balance, + extras)

In [ ]:
applicant = U.parse_applicant_fields(details_lines)
applicant

## 9. PP-StructureV3 — detect tables (layout) on statement pages
First run downloads table models to `%USERPROFILE%\.paddlex\official_models` from Baidu BOS (`paddle-model-ecology.bj.bcebos.com`), not Hugging Face. Cell 0 sets that source.
Formula / chart / seal / unwarping are off for speed. About 7 GB RAM. Close other heavy apps.

In [ ]:
pps = {}
t0 = time.time()
REUSE_CACHE = True   # skip pages already parsed (PP-StructureV3 is ~30-90 s/page on CPU)
for p in statement_pages:
    f = OUT / f"ppstruct_p{p+1}.json"
    if REUSE_CACHE and f.exists():
        pps[p] = json.loads(f.read_text(encoding="utf-8")); continue
    bgr = np.ascontiguousarray(pages_rgb[p][:, :, ::-1])
    pps[p] = U.ppstructure_run(bgr)
    U.save_json(pps[p], OUT / f"ppstruct_p{p+1}.json")
print(f"{time.time()-t0:.1f}s for {len(pps)} pages")

In [ ]:
table_boxes = {p: [b for b in r["layout_det_res"]["boxes"] if b["label"] == "table"] for p, r in pps.items()}
{p+1: [(round(b["score"], 2), [int(x) for x in b["coordinate"]]) for b in bs] for p, bs in table_boxes.items()}

## 10. PP-StructureV3 — table structure + cell OCR → HTML → raw dataframes

In [ ]:
raw_tables = []
for p, r in pps.items():
    for k, t in enumerate(r.get("table_res_list", [])):
        df = U.table_html_to_df(t["pred_html"])
        df.insert(0, "_page", p + 1)
        raw_tables.append(df)
print(len(raw_tables), "tables")
raw_tables[0].head(10) if raw_tables else None

## 11. Normalise columns → Date, Description, Cr/Dr, Amount, Balance

In [ ]:
norm = []
for df in raw_tables:
    page = df["_page"].iloc[0]
    n = U.normalize_columns(df.drop(columns="_page"))
    n["page"] = page
    norm.append(n)
for n in norm:
    print(n["page"].iloc[0] if len(n) else "-", list(n.columns)[:10], len(n))

## 12. Stack pages → months_available

In [ ]:
tx = pd.concat(norm, ignore_index=True) if norm else pd.DataFrame(columns=["date"])
keep = [c for c in ["page", "date", "description", "drcr", "debit", "credit", "amount", "balance"] if c in tx]
tx = tx[keep + [c for c in tx.columns if c not in keep]]
months = U.months_covered(tx["date"]) if "date" in tx else []
print(len(tx), "transactions; months:", months)
tx.head()

## 13. Lead summary + Excel outputs
Full = ≥ 6 months, Partial = 1–5, No = 0. (Img2table comparison for step 10 is a later optional trial.)

In [ ]:
lead = {"lead_id": LEAD_ID, **applicant, **U.banking_flags(len(months)),
        "months_list": ", ".join(months), "n_statement_pages": len(statement_pages), "n_transactions": len(tx)}
tx.insert(0, "lead_id", LEAD_ID)
paths = U.write_outputs(lead, tx, OUT)
print(paths); pd.DataFrame([lead]).T